# Séance 3 — Patterns, MCP et multi-agents
**Agentic AI** · Master 2 I2A · Université Paris 8<br>
**Enseignant :** Pr Mehdi Ammi — Laboratoire LIASD (EA 4383)<br>
**Version :** octobre 2026

*Note sur l'usage de l'IA : le contenu pédagogique (objectifs, progression, exercices, solutions) a été conçu par l'auteur. Claude Code (Anthropic) a été utilisé pour la mise en forme du notebook.*

---

> **🎯 Objectifs de la séance**
> - Reconnaître et implémenter les patterns classiques (routage, évaluateur–optimiseur, parallélisation)
> - Mettre un humain dans la boucle avec `interrupt`
> - Rendre ses outils réutilisables avec un serveur MCP (FastMCP) et les charger dans LangGraph
> - Décider quand plusieurs agents valent leur coût

> **⏱️ Durée totale : 6 h**
> - Partie 1 — Notions : ~1 h
> - Partie 2 — TP guidé puis autonome : ~4 h
> - Livrable de fin de séance : ~50 min

> **📥 Livrable de fin de séance :** à déposer sur **Moodle**, dans l'espace « Livrable 3 » du cours, **à la fin de la journée**. Le sujet se trouve dans la section *Livrable 3*, en fin de notebook.

---

## Prérequis techniques

```bash
# Depuis le dossier seance3
pip install -r requirements.txt
cp .env.example .env          # puis choisir l'option A (Ollama) ou B (API personnelle)
python common/check_setup.py  # doit afficher « OK »

# Option A — LLM local avec Ollama (https://ollama.com)
ollama serve
ollama pull qwen3:8b
```

Pour tester les serveurs MCP : `fastmcp dev <fichier>.py` ouvre l'inspecteur MCP dans le navigateur.

**Données utilisées :** les outils et le corpus des séances 1 et 2 ; pour le livrable, le fichier `agenda.json` fourni.

**Projet fil rouge :** un assistant de veille scientifique qui cherche des articles, les lit, les résume et rédige une synthèse critique. Il s'enrichit à chaque séance.

> **Organisation des travaux pratiques**
> - Les cellules **NE PAS MODIFIER — Démonstration** sont fournies : exécutez-les et lisez le code.
> - Les cellules **✏️ À COMPLÉTER** contiennent des `TODO`.
> - L'accompagnement diminue au fil de la séance. Chaque exercice précise sa **modalité** : compléter le code fourni, implémenter à partir des étapes indiquées, réutiliser et étendre votre propre code, puis concevoir la solution en autonomie à partir des critères de réussite.

> **Indices :** 1 niveau disponible auprès de l'enseignant. Tenez un **journal de décisions** tout au long de la séance.

## Questions de rappel
1. Quelle différence entre RAG classique et RAG agentique ?
2. À quoi sert le `thread_id` ?
3. Dessinez le graphe minimal d'un agent ReAct en LangGraph.

*Vos réponses :*

---
# Partie 1 — Notions
---

**Patterns** (voir [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)) :

| Pattern | Idée | Brique LangGraph |
| --- | --- | --- |
| Enchaînement de prompts | Étapes fixes, la sortie de l'une nourrit la suivante | Arêtes fixes |
| Routage | Un classifieur choisit la branche | Arête conditionnelle |
| Parallélisation | Plusieurs sous-tâches en même temps | `Send` |
| Orchestrateur–travailleurs | Un LLM découpe la tâche et répartit | `Send` + agrégation |
| Évaluateur–optimiseur | Un LLM produit, un autre critique, on boucle | Cycle + compteur |

**Validation humaine :** `interrupt()` met le graphe en pause ; on reprend avec `Command(resume=...)`. Le checkpointer est obligatoire.

**MCP (Model Context Protocol) :** un protocole standard entre une application (client) et un serveur qui expose des **outils**, des **ressources** et des **prompts**. Un même serveur sert plusieurs agents et applications. Dans LangGraph : `langchain-mcp-adapters`.

**Multi-agents :** superviseur et spécialistes, sous-graphes, passage de relais avec `Command(goto=...)`. Chaque agent a son propre contexte.
- Vaut son coût : tâches séparables, contextes longs, consignes très différentes.
- Ne vaut pas son coût : latence, nombre d'appels multiplié, erreurs qui se propagent d'un agent à l'autre.

---
# Partie 2 — Travaux pratiques
---

## Exercice 1 — Choix d'une architecture

> **Modalité :** répondre par écrit, puis correction collective.

Pour chaque scénario : workflow ou agent ? Quel pattern ? Justifiez en une ligne. Correction collective.

| # | Scénario | Workflow ou agent ? | Pattern | Justification |
| --- | --- | --- | --- | --- |
| 1 | Traduire un article puis en extraire 5 mots-clés | | | |
| 2 | Répondre aux mails d'étudiants : scolarité, technique ou pédagogique | | | |
| 3 | Résumer 40 articles pour une revue de littérature | | | |
| 4 | Écrire une synthèse puis l'améliorer jusqu'à atteindre une qualité donnée | | | |
| 5 | Déboguer un dépôt de code inconnu | | | |
| 6 | Planifier un voyage avec vols, hôtel et budget à respecter | | | |

---
## Exercice 2 — Premier serveur MCP

> **Modalité :** compléter le code fourni.

1. Lancez l'inspecteur sur le serveur de démonstration : `fastmcp dev mcp_server_demo.py`, et appelez l'outil.
2. Chargez cet outil dans un agent avec la cellule de démonstration.
3. Complétez ensuite les deux cellules suivantes : elles créent **votre** serveur MCP à partir de votre `search_arxiv` de la séance 1, puis le chargent.

In [ ]:
# NE PAS MODIFIER — Démonstration
# Le dossier common/ se trouve dans le dossier de la séance
from common.llm import get_llm
llm = get_llm()

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

client_mcp = MultiServerMCPClient({
    "demo": {"command": "python", "args": ["mcp_server_demo.py"], "transport": "stdio"},
})
outils_mcp = await client_mcp.get_tools()
print([o.name for o in outils_mcp])

agent = create_agent(model=llm, tools=outils_mcp)
r = await agent.ainvoke({"messages": [("user", "Combien de mots dans : « les agents utilisent des outils » ?")]})
print(r["messages"][-1].content)

In [ ]:
# ✏️ À COMPLÉTER
%%writefile veille_server.py
import arxiv
from fastmcp import FastMCP

mcp = FastMCP("veille")


@mcp.tool
def search_arxiv(query: str, max_results: int = 5) -> list[dict]:
    """TODO : reprendre la docstring de votre outil de la séance 1."""
    ...  # TODO : reprendre le corps de votre search_arxiv de la séance 1


if __name__ == "__main__":
    mcp.run()

In [ ]:
# ✏️ À COMPLÉTER
client_veille = MultiServerMCPClient({
    "veille": {...},  # TODO : sur le modèle du serveur "demo" ci-dessus
})
outils_veille = await client_veille.get_tools()
print([o.name for o in outils_veille])

---
## Exercice 3 — Boucle évaluateur–optimiseur

> **Modalité :** compléter le code fourni.

Un nœud rédige, un nœud critique, et on boucle tant que la qualité est insuffisante. Complétez les trois parties marquées `TODO`.

```
START → rediger → critiquer ──(note ≥ 8 ou 3 tours ?)──oui──→ END
            ↑__________________________non___|
```

In [ ]:
# ✏️ À COMPLÉTER
from typing import TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END


class Critique(BaseModel):
    note: int = Field(ge=1, le=10, description="Qualité de la synthèse, de 1 à 10")
    remarques: str = Field(description="Ce qu'il faut améliorer, en 2 ou 3 points")


class EtatSynthese(TypedDict):
    sujet: str
    brouillon: str
    critique: dict | None   # la critique, convertie en dict (sauvegardable par le checkpointer)
    tours: int


def rediger(etat: EtatSynthese) -> dict:
    consigne = f"Rédige une synthèse de 5 lignes sur : {etat['sujet']}."
    if etat["critique"]:
        consigne += f"\nAméliore ce brouillon : {etat['brouillon']}\nRemarques : {etat['critique']['remarques']}"
    return {"brouillon": llm.invoke(consigne).content, "tours": etat["tours"] + 1}


critique_llm = llm.with_structured_output(Critique)


def critiquer(etat: EtatSynthese) -> dict:
    return {"critique": ...}  # TODO 1 : faire noter etat["brouillon"] par critique_llm, puis .model_dump()


def continuer(etat: EtatSynthese) -> str:
    ...  # TODO 2 : renvoyer "fin" si la note est >= 8 ou si 3 tours sont faits, sinon "rediger"


g = StateGraph(EtatSynthese)
g.add_node("rediger", rediger)
g.add_node("critiquer", critiquer)
g.add_edge(START, "rediger")
g.add_edge("rediger", "critiquer")
g.add_conditional_edges("critiquer", continuer, {...})  # TODO 3 : associer "rediger" et "fin" aux nœuds
boucle = g.compile()

r = boucle.invoke({"sujet": "les agents LLM pour la veille scientifique",
                   "brouillon": "", "critique": None, "tours": 0})
print(r["tours"], "tours —", r["critique"])
print(r["brouillon"])

## Exercice 4 — Validation humaine

> **Modalité :** implémenter à partir des étapes indiquées.

Ajoutez un nœud `valider` entre la critique et la fin. Les étapes sont données ; le code est à écrire.

1. Dans `valider`, appelez `interrupt(...)` avec le brouillon : le graphe se met en pause, et la valeur renvoyée par `interrupt` sera la réponse de l'humain.
2. Si la réponse est `"ok"`, allez à `END` ; sinon, rangez la remarque de l'humain dans la critique et retournez à `rediger` avec `Command(goto=...)`.
3. Reconstruisez le graphe de l'exercice 3 avec ce nœud et **un checkpointer** (obligatoire pour reprendre).
4. Lancez le graphe, lisez `r["__interrupt__"]`, puis reprenez avec `Command(resume="ok")` ou une remarque.

Documentation : [Human-in-the-loop](https://docs.langchain.com/oss/python/langgraph/interrupts)

In [ ]:
# ✏️ À COMPLÉTER
from typing import Literal
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


def valider(etat: EtatSynthese) -> Command[Literal["rediger", "__end__"]]:
    ...


# Reconstruire le graphe ici, puis :
# config = {"configurable": {"thread_id": "1"}}
# r = boucle_validee.invoke({...}, config)
# print(r["__interrupt__"])
# r = boucle_validee.invoke(Command(resume="ok"), config)

---
# Travail en autonomie

Les exercices suivants s'appuient sur le code que vous avez déjà écrit.

## Exercice 5 — Serveur MCP complet

> **Modalité :** implémenter à partir des étapes indiquées.

Ajoutez à `veille_server.py` vos outils `read_pdf` (séance 1) et `search_corpus` (séance 2).
1. Copiez chaque fonction dans le serveur et décorez-la avec `@mcp.tool`.
2. Pour `search_corpus`, ouvrez la collection Chroma de la séance 2 au démarrage du serveur.
3. Vérifiez les 3 outils dans l'inspecteur, puis rechargez-les dans le notebook.

**Critères :** les 3 outils sont visibles dans l'inspecteur ; une erreur côté serveur renvoie un message lisible sans faire planter l'agent.

Documentation : [FastMCP](https://gofastmcp.com) · [langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters)

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 5 — complétez veille_server.py (dans le fichier ou avec %%writefile)

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 5 — rechargez les outils

## Exercice 6 — Agent critique en sous-graphe

> **Modalité :** réutiliser et étendre votre propre code.

Reprenez la boucle des exercices 3 et 4 et intégrez-la à votre graphe sous forme de **sous-graphe** : l'agent critique a ses propres consignes et ne voit que la synthèse et les sources, pas tout l'historique.

**Critères :** 2 tours d'échange au maximum ; les critiques apparaissent dans les traces ; validation humaine avant la synthèse finale.

Documentation : [Multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent)

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 6

## Exercice 7 — Analyse coût–bénéfice

> **Modalité :** réutiliser et étendre votre propre code.

Comparez votre graphe **sans** et **avec** l'agent critique de l'exercice 6, sur 5 thèmes de veille.

| Version | Qualité perçue (1–5) | Nombre d'appels LLM | Temps total | Décision |
| --- | --- | --- | --- | --- |
| Sans critique | | | | |
| Avec critique | | | | |

Quelle version gardez-vous, et pourquoi ?

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 7

## Exercice 8 — Projet fil rouge

> **Modalité :** concevoir la solution en autonomie, à partir des critères de réussite.

Assemblez l'assistant complet : votre graphe de la séance 2, tous ses outils servis par votre serveur MCP (exercice 5), la critique et la validation humaine (exercice 6).

**Critères :** plus aucun outil n'est défini dans le notebook, la synthèse finale passe toujours par la validation humaine, l'assistant s'arrête seul.

In [ ]:
# ✏️ À COMPLÉTER
# Exercice 8

### Journal de décisions
| Décision | Alternatives écartées | Raison |
| --- | --- | --- |
| | | |

---
# Livrable 3 — Planificateur de révisions

> **📥 Dépôt sur Moodle — à la fin de la journée**<br>
> Déposez ce notebook exécuté (`S3_patterns_mcp_multiagents.ipynb`), votre fichier `agenda_server.py` et la réponse écrite dans l'espace **« Livrable 3 »** du cours sur Moodle.


**Sujet.**
1. Complétez `agenda_server.py` : outils `list_events` et `add_event` sur `agenda.json`.
2. Construisez un graphe LangGraph qui charge ces outils via MCP, propose un planning de révisions pour la semaine du 16 novembre 2026 (10 h de révision, sans chevauchement), le fait **critiquer** (2 tours au maximum), puis s'arrête avec `interrupt` pour **validation avant tout appel à `add_event`**.
3. Répondez en 10 lignes au scénario ci-dessous.

> *Scénario :* une scolarité veut un assistant qui répond aux questions des étudiants, vérifie leur dossier dans le système d'information et prépare des attestations. Faut-il un agent unique ou plusieurs agents ?

**Critères de réussite :** les outils sont visibles dans l'inspecteur MCP, aucun événement n'est ajouté sans validation, la boucle de critique s'arrête bien, et la justification s'appuie sur le coût, la latence et la séparation des tâches.

In [ ]:
# ✏️ À COMPLÉTER
# Livrable 3 — complétez agenda_server.py (dans le fichier ou avec %%writefile)

In [ ]:
# ✏️ À COMPLÉTER
# Livrable 3 — votre graphe

*Réponse au scénario (10 lignes) :*

---
# Exercices d'approfondissement

1. Résumez N articles **en parallèle** avec `Send`, puis mesurez le gain de temps.
2. Construisez un **superviseur** qui coordonne trois agents (chercheur, rédacteur, critique).
3. Branchez votre serveur MCP dans un **client MCP existant** et utilisez-le hors de votre code.
4. Exposez un **prompt MCP** réutilisable (par exemple « synthèse critique ») en plus des outils.

In [ ]:
# ✏️ À COMPLÉTER
# Exercices d'approfondissement

---
## Journal de bord

- Un échec observé aujourd'hui :
- Sa cause :
- Ce que j'ai fait pour le corriger :